In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import lightgbm as lgb
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [8]:
BASE_DIR = Path("E:/Code/Poliforge/Polyforge-AI/Polyforge-AI")
DATA_PATH = BASE_DIR / "data" / "raw" / "polymers_with_names_predicted_selfies.csv"
df = pd.read_csv(DATA_PATH)

# Свойства для предсказания
property_cols = [c for c in df.columns if c not in ['ID','Name','SELFIES','Polymer_SMILES',
                 'Monomer_SMILES_1','Monomer_SMILES_2']]

# Новый MorganGenerator (радиус 2, 1024 бит)
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def smiles_to_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(1024)
    return np.array(gen.GetFingerprint(mol))

X = np.array([smiles_to_fingerprint(s) for s in df['Polymer_SMILES']])
y = df[property_cols].values

print(f"Размерность X: {X.shape}, y: {y.shape}")

Размерность X: (11239, 1024), y: (11239, 36)


In [9]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

# Масштабируем целевые переменные для улучшения обучения
scaler = StandardScaler()
y_train_scaled = scaler.fit_transform(y_train)
y_val_scaled = scaler.transform(y_val)

In [10]:
def train_lgbm_gpu(X_tr, y_tr, X_vl, y_vl, prop_cols, scaler):
    # Параметры LightGBM
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'random_state': 42,
        'device': 'gpu',          # GPU
        'gpu_platform_id': 0,
        'gpu_device_id': 0,
    }
    try:
        # Пробуем GPU
        model = lgb.LGBMRegressor(**params, n_estimators=500, early_stopping_round=50)
        model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)])
        print("✅ LightGBM успешно обучена на GPU")
    except Exception as e:
        print(f"⚠️ Ошибка GPU LightGBM: {e}")
        print("Переключаемся на CPU LightGBM...")
        params['device'] = 'cpu'
        model = lgb.LGBMRegressor(**params, n_estimators=500, early_stopping_round=50)
        model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)])
    return model

# Обучаем отдельную модель для каждого свойства (37 моделей)
models = {}
val_preds = np.zeros_like(y_val)
train_preds = np.zeros_like(y_train)

for i, prop in enumerate(property_cols):
    print(f"\nОбучение для свойства {prop} ({i+1}/{len(property_cols)})")
    model = train_lgbm_gpu(X_train, y_train_scaled[:, i], X_val, y_val_scaled[:, i],
                           [prop], scaler)  # scaler не нужен отдельно, но для совместимости
    models[prop] = model
    # Предсказания (в скейлированном виде)
    train_preds[:, i] = model.predict(X_train)
    val_preds[:, i] = model.predict(X_val)

# Обратное масштабирование
train_preds = scaler.inverse_transform(train_preds)
val_preds = scaler.inverse_transform(val_preds)


Обучение для свойства Egc (1/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства Egb (2/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства Eib (3/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства CED (4/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства Ei (5/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства Eea (6/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства nc (7/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства ne (8/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства Xc (9/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства Xe (10/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства epse_6.0 (11/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства epsc (12/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства epse_3.0 (13/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства epse_1.78 (14/36)
✅ LightGBM успешно обучена на GPU

Обучение для свойства epse_15.

In [11]:
train_mae = mean_absolute_error(y_train, train_preds, multioutput='uniform_average')
val_mae = mean_absolute_error(y_val, val_preds, multioutput='uniform_average')
train_r2 = r2_score(y_train, train_preds, multioutput='uniform_average')
val_r2 = r2_score(y_val, val_preds, multioutput='uniform_average')

print(f"\n📊 LightGBM итоговые метрики:")
print(f"  Train MAE: {train_mae:.4f}, R2: {train_r2:.4f}")
print(f"  Val   MAE: {val_mae:.4f}, R2: {val_r2:.4f}")

print("\n   MAE по свойствам (валидация):")
for i, col in enumerate(property_cols):
    mae = mean_absolute_error(y_val[:, i], val_preds[:, i])
    r2 = r2_score(y_val[:, i], val_preds[:, i])
    print(f"    {col:20s}: MAE={mae:8.4f}, R2={r2:.4f}")


📊 LightGBM итоговые метрики:
  Train MAE: 4.9828, R2: 0.9439
  Val   MAE: 6.4738, R2: 0.8918

   MAE по свойствам (валидация):
    Egc                 : MAE=  0.1672, R2=0.9295
    Egb                 : MAE=  0.1748, R2=0.9382
    Eib                 : MAE=  0.0643, R2=0.9353
    CED                 : MAE=  3.9721, R2=0.8901
    Ei                  : MAE=  0.1144, R2=0.8897
    Eea                 : MAE=  0.1049, R2=0.9268
    nc                  : MAE=  0.0238, R2=0.9229
    ne                  : MAE=  0.0140, R2=0.9285
    Xc                  : MAE=  2.8839, R2=0.8284
    Xe                  : MAE=  2.2494, R2=0.8299
    epse_6.0            : MAE=  0.1048, R2=0.8729
    epsc                : MAE=  0.1131, R2=0.8911
    epse_3.0            : MAE=  0.1256, R2=0.8689
    epse_1.78           : MAE=  0.1367, R2=0.8782
    epse_15.0           : MAE=  0.0383, R2=0.8938
    epse_4.0            : MAE=  0.1173, R2=0.8705
    epse_5.0            : MAE=  0.1103, R2=0.8728
    epse_2.0          

In [14]:
import joblib

MODEL_SAVE_PATH = BASE_DIR / "models" / "fingerprint_property_model.pkl"
feature_names = [f"fp_{i}" for i in range(X_train.shape[1])]  # fp_0, fp_1, ...

joblib.dump({
    'models': models,
    'scaler': scaler,
    'property_cols': property_cols,
    'feature_names': feature_names      # <-- добавляем
}, MODEL_SAVE_PATH)

['E:\\Code\\Poliforge\\Polyforge-AI\\Polyforge-AI\\models\\fingerprint_property_model.pkl']